# 01 · Clean the unified dataset

**Shared pipeline notebook (team-owned).** Run it after `00_build_dataset.ipynb`. Changes go through a PR.

**Input:** `data/clean/films_unified.csv` (one row per film, every source's columns side by side).
**Outputs** (gitignored; rebuilt on every run):
- `data/clean/films_clean.csv`: same films and rows, plus one **combined column per fact** (runtime, US release date, rating, genre…), with a `_src` column showing which source supplied the value and a `_missing` flag for inputs that aren't always filled.
- `data/clean/films_clean_columns.csv`: a dictionary giving every column's group and role.

**Why:** each fact (runtime, genre, rating…) comes from several sources, each in its own column. The rules, agreed in `Edwards-A10.ipynb` §4b (Tanner, 2026-11-24), are:
1. Each fact gets **one combined column**, filled from the fullest source first, then from the others in a fixed order.
2. Only real values count. TMDB's `0` for budget, revenue or runtime means unknown, and so does Box Office Mojo's `-` distributor.
3. Values are put into one format before combining (rating codes, genre names, language codes, country names).
4. The original source columns are **kept but grouped as `source detail`**: there for audits, never model inputs.
5. Inputs that aren't always filled get a `_missing` flag. Blanks aren't guessed or dropped, because coverage depends on budget (Data Understanding §3h).

**This notebook never removes films.** Later cleaning steps (U3 revenue repair, U11 junk entries, standardizing company names) will be added here as numbered sections.

## 1. Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

CLEAN = Path("../data/clean")
films = pd.read_csv(CLEAN / "films_unified.csv", low_memory=False)
assert films["film_uid"].is_unique
n_rows = len(films)
usable = films["usable"]
print(f"films_unified.csv: {n_rows:,} films x {films.shape[1]} columns ({usable.sum():,} usable)")

## 2. Helpers

- `real(col)`: the column with placeholder values turned into blanks.
- `combine(name, [cols])`: the first real value across the columns, in order, plus `<name>_src` naming the source column it came from.

In [ ]:
ZERO_MISSING = {"budget", "revenue", "runtime", "tn_Domestic Gross (USD)"}   # 0 = unknown in these sources
DASH_MISSING = {"bom_distributor"}


def real(c):
    s = films[c]
    if c in ZERO_MISSING:
        s = s.where(s.ne(0))
    if s.dtype == object or str(s.dtype) == "str":
        s = s.where(s.astype(str).str.strip().ne(""))
    if c in DASH_MISSING:
        s = s.where(s.astype(str).str.strip().ne("-"))
    return s


def combine(name, cols, values=None):
    """First real value across `cols` (in order). `values` optionally supplies already-formatted Series per col."""
    vals = pd.concat([(values or {}).get(c, real(c)).rename(c) for c in cols], axis=1)
    src = pd.Series(pd.NA, index=films.index, dtype="string")
    for c in reversed(cols):
        src = src.mask(vals[c].notna(), c)
    films[name] = vals.bfill(axis=1).iloc[:, 0].infer_objects() if len(cols) > 1 else vals.iloc[:, 0]
    films[name + "_src"] = src
    u = films.loc[usable, name]
    print(f"{name:<22} usable films filled {u.notna().mean():6.1%}   by source: "
          + ", ".join(f"{k} {v:.0%}" for k, v in films.loc[usable, name + '_src'].value_counts(normalize=True).items()))


def split_list(s, sep=","):
    return s.fillna("").astype(str).map(lambda x: [t.strip() for t in x.split(sep) if t.strip()])


def join_list(lists):
    return lists.map(lambda l: ", ".join(dict.fromkeys(l)) if l else np.nan)

## 3. Money (C1–C3)

`fin_budget` and `fin_revenue` are already combined by the build (TMDB → The Numbers → Wikidata) and are kept unchanged. `domestic_gross` is new: The Numbers → Box Office Mojo → RT. It's post-release, so it's **for checks only** (S4, U3), never an input. A The Numbers domestic gross of 0 counts as unknown.

In [ ]:
combine("domestic_gross", ["tn_Domestic Gross (USD)", "bom_domestic_gross", "rt_boxOffice_usd"])

## 4. Runtime and release date (C4–C6)

- `runtime_min`: TMDB → RT → The Numbers → RT sl (sources agree on 92–93% of films within 5 minutes).
- `release_date_us`: **the US theatrical date** (decision, Tanner 2026-11-24): The Numbers → RT theatrical → Box Office Mojo first day.
- `release_date`: US date where known, otherwise TMDB's (first release anywhere). `release_date_src` records which. `release_month` and `release_quarter` come from it.
- `year` (TMDB) is **not changed**: the population and the train/test split stay as the contract defines them.

In [ ]:
combine("runtime_min", ["runtime", "rt_runtimeMinutes", "tn_Running Time (minutes)", "sl_runtime"])

dates = {c: pd.to_datetime(films[c], errors="coerce") for c in
         ["tn_Release Date", "rt_releaseDateTheaters", "bom_first_date", "release_date"]}
combine("release_date_us", ["tn_Release Date", "rt_releaseDateTheaters", "bom_first_date"], dates)
dates["release_date_us"] = films["release_date_us"]
films["release_date_tmdb"] = dates["release_date"]
combine("release_date", ["release_date_us", "release_date"], {"release_date_us": dates["release_date_us"], "release_date": dates["release_date"]})
films["release_date_src"] = films["release_date_src"].replace({"release_date_us": "us", "release_date": "tmdb_first_release"})
films["release_month"] = films["release_date"].dt.month
films["release_quarter"] = films["release_date"].dt.quarter
films = films.drop(columns="release_date_us_src")

## 5. Genre (C7)

TMDB genres (19 categories) first. RT and The Numbers genres fill only TMDB's blanks, translated to TMDB's names. Genres with no TMDB equivalent (*Biography*, *Sports*, *Holiday*, *LGBTQ+*, *Multiple Genres*) are left out of the fill.

In [ ]:
RT_GENRE = {"Action": "Action", "Adventure": "Adventure", "Animation": "Animation", "Anime": "Animation",
            "Comedy": "Comedy", "Stand-up": "Comedy", "Crime": "Crime", "Documentary": "Documentary", "Drama": "Drama",
            "Fantasy": "Fantasy", "History": "History", "Horror": "Horror", "Kids & family": "Family",
            "Music": "Music", "Musical": "Music", "Mystery & thriller": ["Mystery", "Thriller"], "Romance": "Romance",
            "Sci-fi": "Science Fiction", "War": "War", "Western": "Western"}
TN_GENRE = {"Action": "Action", "Adventure": "Adventure", "Black Comedy": "Comedy", "Comedy": "Comedy",
            "Concert/Performance": "Music", "Documentary": "Documentary", "Drama": "Drama", "Horror": "Horror",
            "Musical": "Music", "Reality": "Documentary", "Romantic Comedy": ["Romance", "Comedy"],
            "Thriller/Suspense": "Thriller", "Western": "Western"}


def translate(s, mapping):
    def one(lst):
        out = []
        for g in lst:
            m = mapping.get(g)
            out += m if isinstance(m, list) else ([m] if m else [])
        return out
    return join_list(split_list(s).map(one))


combine("genres_all", ["genres", "rt_genre", "tn_Genre"],
        {"genres": real("genres"), "rt_genre": translate(films["rt_genre"], RT_GENRE),
         "tn_Genre": translate(films["tn_Genre"].astype("string").str.strip(), TN_GENRE)})
TMDB_GENRES = set(g for l in split_list(films["genres"]) for g in l)
assert set(g for l in split_list(films["genres_all"]) for g in l) <= TMDB_GENRES

## 6. MPAA rating (C8)

The Numbers → RT → RT sl, standardized to **G / PG / PG-13 / R / NC-17 / NR** (sources agree on 99% of films). The Numbers' run-on text ("NC-17for explicit sexual content…") is cut to its leading rating; "Not Rated" and "NR" become NR; TV ratings and "Open" count as missing.

⚠️ RT never uses NR, so a blank RT rating can mean "never rated". `mpaa_rating` NR and missing are kept separate.

In [ ]:
def std_rating(s):
    s = s.astype("string").str.strip()
    m = s.str.extract(r"^(NC-?17|PG-13|PG|G|R|Not Rated|NR|Unrated)", expand=False)
    return m.replace({"NC17": "NC-17", "Not Rated": "NR", "Unrated": "NR"})


combine("mpaa_rating", ["tn_MPAA Rating", "rt_rating", "sl_content_rating"],
        {c: std_rating(films[c]) for c in ["tn_MPAA Rating", "rt_rating", "sl_content_rating"]})
films.loc[usable, "mpaa_rating"].value_counts(dropna=False)

## 7. People and companies (C9–C13)

Fill only: the second source fills the first source's blanks, with no name matching yet.
- `director`, `writer`: RT → RT sl. They're used as **track records from earlier films only**.
- `production_companies`: TMDB → The Numbers → RT sl.
- `distributor`: RT → Box Office Mojo.

**Cast (C11) is dropped as an input** (Tanner, 2026-11-24): only 5% coverage in 2020–23. `sl_actors` stays in source detail.

Company and distributor names differ by source ("Warner Bros." vs "Warner Bros. Pictures"). Standardizing them is a later cleaning step.

In [ ]:
combine("director", ["rt_director", "sl_directors"])
combine("writer", ["rt_writer", "sl_authors"])
combine("production_companies", ["production_companies", "tn_Production/Financing Companies", "sl_production_company"],
        {"production_companies": real("production_companies"),
         "tn_Production/Financing Companies": join_list(split_list(films["tn_Production/Financing Companies"])),
         "sl_production_company": real("sl_production_company")})
combine("distributor", ["rt_distributor", "bom_distributor"])

## 8. Country and language (C14–C15)

- `countries`: TMDB → The Numbers, with The Numbers' names translated to TMDB's ("United States" → "United States of America"). The Numbers writes Taiwan as "Taiwan, Province of China", which splits on the comma; the "Province of China" fragment is dropped.
- `original_language`: TMDB (ISO code) → RT (language name → code). `tn_Languages` lists every spoken language, not the original one, so it stays in source detail.

In [ ]:
TN_COUNTRY = {"United States": "United States of America", "Republic of Korea": "South Korea",
              "Russian Federation": "Russia", "Islamic Republic of Iran": "Iran",
              "Bolivarian Republic of Venezuela": "Venezuela", "Province of China": None}
tn_c = join_list(split_list(films["tn_Production Countries"]).map(
    lambda l: [TN_COUNTRY.get(c, c) for c in l if TN_COUNTRY.get(c, c)]))
combine("countries", ["production_countries", "tn_Production Countries"],
        {"production_countries": real("production_countries"), "tn_Production Countries": tn_c})

LANG = {"English": "en", "Hindi": "hi", "French": "fr", "Chinese": "zh", "Korean": "ko", "Spanish": "es",
        "Russian": "ru", "Japanese": "ja", "German": "de", "Italian": "it", "Arabic": "ar", "Norwegian": "no",
        "Tamil": "ta", "Danish": "da", "Telugu": "te", "Portuguese": "pt", "Turkish": "tr", "Polish": "pl",
        "Finnish": "fi", "Swedish": "sv", "Dutch": "nl", "Malayalam": "ml", "Hebrew": "he", "Romanian": "ro",
        "Czech": "cs", "Thai": "th", "Urdu": "ur", "Indonesian": "id", "Persian": "fa", "Malay": "ms",
        "Latin": "la", "Hungarian": "hu", "Afrikaans": "af", "Cantonese": "cn", "Greek": "el", "Icelandic": "is",
        "Vietnamese": "vi", "Bengali": "bn", "Kannada": "kn", "Marathi": "mr", "Punjabi": "pa", "Tagalog": "tl",
        "Ukrainian": "uk", "Serbian": "sr", "Croatian": "hr", "Estonian": "et", "Lithuanian": "lt", "Slovak": "sk"}
rt_lang = films["rt_originalLanguage"].astype("string").str.replace(r"\s*\(.*\)", "", regex=True).str.strip().map(LANG)
combine("original_language", ["original_language", "rt_originalLanguage"],
        {"original_language": real("original_language"), "rt_originalLanguage": rt_lang})

## 9. Review consensus and audience score (C16–C17): description only

Both are **post-release**, so they're never model inputs. They're kept to describe films and to show how reception relates to money-losers (Tanner, 2026-11-24).
- `critic_pct_fresh` (0–100): RT reviews % fresh → RT Tomatometer → RT sl Tomatometer.
- `audience_score` (0–100): TMDB average vote × 10 (only when the film has votes) → RT audience → RT sl audience.

In [ ]:
combine("critic_pct_fresh", ["rtrev_pct_fresh", "rt_tomatoMeter", "sl_tomatometer_rating"])
tmdb_aud = (films["vote_average"] * 10).where(films["vote_count"].gt(0))
combine("audience_score", ["vote_average", "rt_audienceScore", "sl_audience_rating"], {"vote_average": tmdb_aud})

## 10. Franchise, source material, creative type (C18)

These come from The Numbers only, so there's nothing to combine. Franchise names that were run together ("BatmanDark Knight Trilogy") are split at a lowercase→uppercase boundary into a `|`-separated list (U9). Names that are legitimately written that way (*MonsterVerse*, *xXx*) are protected.

In [ ]:
PROTECT = ["MonsterVerse", "xXx", "DreamWorks", "McG", "LEGO", "iCarly"]


def split_franchise(x):
    if pd.isna(x):
        return np.nan
    s = str(x)
    for i, p in enumerate(PROTECT):
        s = s.replace(p, f"\x00{i}\x00")
    s = __import__("re").sub(r"(?<=[a-z0-9’'\)])(?=[A-Z])", "|", s)
    for i, p in enumerate(PROTECT):
        s = s.replace(f"\x00{i}\x00", p)
    return s


films["franchise"] = films["tn_Franchise"].map(split_franchise)
films["source_material"] = real("tn_Source")
films["creative_type"] = real("tn_Creative Type")
changed = films.loc[films["franchise"].ne(films["tn_Franchise"]) & films["tn_Franchise"].notna(), ["tn_Franchise", "franchise"]]
print(f"Split {len(changed)} run-together franchise values (rows, all films)")
changed.drop_duplicates().head(10)

## 11. Missing flags (U6)

Every allowed or candidate input that isn't filled for all usable films gets `<column>_missing` (True = no source had a value). Models use the flag instead of a guessed value.

In [ ]:
INPUTS = ["runtime_min", "release_date", "genres_all", "mpaa_rating", "director", "writer", "production_companies",
          "distributor", "countries", "original_language", "franchise", "source_material", "creative_type"]
flags = {}
for c in INPUTS:
    miss = films[c].isna()
    if miss[usable].any():
        films[c + "_missing"] = miss
        flags[c] = miss[usable].mean()
pd.Series(flags, name="share of usable films missing").map("{:.1%}".format).to_frame()

## 12. Column groups and save

Columns are reordered into groups, and every column's group and role is written to `films_clean_columns.csv`:

| Group | What | Model input? |
|---|---|---|
| `key` | `film_uid`, ids, titles | no |
| `population` | `year`, `usable`, `in_tmdb`, `merged_ids` | no (defines rows and splits) |
| `money` | `fin_budget` (input), `fin_revenue` (label), `domestic_gross` (checks) | budget only |
| `input` | the combined columns above, plus `_missing` flags | yes, subject to the contract (A4 for candidates) |
| `provenance` | `_src` columns | no |
| `description` | `critic_pct_fresh`, `audience_score`, review counts, critic consensus text | no (post-release) |
| `source detail` | every original source column | no |

In [ ]:
KEY = ["film_uid", "id", "imdb_id", "title", "original_title"]
POPULATION = ["year", "usable", "in_tmdb", "merged_ids"]
MONEY = ["fin_budget", "fin_budget_src", "fin_revenue", "fin_revenue_src", "domestic_gross"]
CANDIDATE = {"mpaa_rating", "writer", "distributor", "franchise", "source_material", "creative_type"}
INPUT = ["runtime_min", "release_date", "release_month", "release_quarter", "release_date_us", "release_date_tmdb",
         "genres_all", "mpaa_rating", "director", "writer", "production_companies", "distributor", "countries",
         "original_language", "franchise", "source_material", "creative_type"]
INPUT += [c + "_missing" for c in INPUTS if c + "_missing" in films]
PROVENANCE = [c for c in films.columns if c.endswith("_src") and c not in MONEY]
DESCRIPTION = ["critic_pct_fresh", "audience_score", "rtrev_n_reviews", "rtrev_n_top_critic", "rtrev_n_before_release",
               "slrev_n_reviews", "slrev_n_before_release", "sl_critics_consensus"]
groups = {"key": KEY, "population": POPULATION, "money": MONEY, "input": INPUT,
          "provenance": PROVENANCE, "description": DESCRIPTION}
placed = [c for cols in groups.values() for c in cols]
groups["source detail"] = [c for c in films.columns if c not in placed]
assert len(set(placed)) == len(placed) and all(c in films for c in placed)

def role(c, g):
    if g == "input":
        base = c.removesuffix("_missing")
        return "candidate input (needs amendment A4)" if base in CANDIDATE else "allowed input"
    if c == "fin_budget":
        return "allowed input"
    if c == "fin_revenue":
        return "label (outcome)"
    if c.endswith("_src"):
        return "build metadata"
    return {"key": "key", "population": "defines rows / splits", "money": "post-release: checks only",
            "provenance": "build metadata", "description": "post-release: description only",
            "source detail": "not used (audit only)"}[g]

dictionary = pd.DataFrame([{"column": c, "group": g, "role": role(c, g),
                            "% filled (usable films)": round(100 * films.loc[usable, c].notna().mean(), 1)}
                           for g, cols in groups.items() for c in cols])
clean = films[dictionary["column"].tolist()]
assert len(clean) == n_rows and clean["film_uid"].is_unique
clean.to_csv(CLEAN / "films_clean.csv", index=False)
dictionary.to_csv(CLEAN / "films_clean_columns.csv", index=False)
print(f"films_clean.csv: {len(clean):,} films x {clean.shape[1]} columns ({clean['usable'].sum():,} usable)")
dictionary.groupby("group", sort=False).size().rename("columns").to_frame()

In [ ]:
with pd.option_context("display.max_rows", 80):
    display(dictionary[dictionary["group"].isin(["money", "input", "description"])])